# Lecture 1: Strings and the Genome

**Course:** Intro to Programming for Computational Biology  
**Prerequisites:** Variables, loops, if/else statements

---
## The Central Dogma of Molecular Biology

All living things store their instructions in **DNA**. These instructions are read out in two steps:

```
Genome (DNA)  →  transcription  →  RNA  →  translation  →  Protein
```

- **DNA** is made of four bases: A, T, G, C — and lives in the nucleus
- **Transcription** copies a gene into messenger RNA (mRNA), replacing T with U
- **Translation** reads the mRNA three bases at a time (codons) to build a protein
- **Proteins** do most of the work in a cell: enzymes, structural components, signals

The part of a gene that actually gets translated into protein is called the **coding sequence (CDS)**.

---
## Why Does This Matter? The Story of HBB

Our example today is **HBB** — the gene encoding **β-globin**, one of the two proteins that make up hemoglobin, the molecule that carries oxygen in your blood.

A single letter change in HBB — one nucleotide out of ~1,600 — causes **sickle cell disease**, a serious blood disorder affecting millions of people worldwide. This mutation changes one amino acid in the protein, which causes hemoglobin to form rigid rods that deform red blood cells into a sickle shape.

By the end of this notebook, you will be able to:
- Manipulate DNA sequences as Python strings
- Write a function that translates DNA into protein
- Use Biopython to work with the real HBB sequence
- Simulate the sickle cell mutation and predict its effect

---
## Section 1: DNA as a Python String

DNA is made of four nucleotide bases: **A** (adenine), **T** (thymine), **G** (guanine), and **C** (cytosine). A DNA sequence is just a string of these letters — which means we can use all our Python string tools on it!

In [ ]:
# The first 30 nucleotides of the HBB coding sequence
dna = "ATGGTGCATCTGACTCCTGAGGAGAAGTCT"
print("Sequence:", dna)
print("Length:", len(dna))

### 1.1 Indexing and slicing

Like any Python string, we can index into a DNA sequence. Remember: Python uses **0-based indexing**.

In [ ]:
# What is the first nucleotide?
print(dna[0])

# What are the first 3 nucleotides (the first codon)?
print(dna[0:3])

# What is the last nucleotide?
print(dna[-1])

**Exercise 1.1** — Fill in the blanks to extract the **4th codon** (nucleotides 10–12, 1-based) from `dna`.

In [ ]:
# Hint: the 4th codon starts at index 9 (0-based)
fourth_codon = dna[___:___]
print("4th codon:", fourth_codon)  # Expected: CTG

### 1.2 Counting nucleotides

Python strings have a built-in `.count()` method.

In [ ]:
# Count each nucleotide
print("A:", dna.count("A"))
print("T:", dna.count("T"))
print("G:", dna.count("G"))
print("C:", dna.count("C"))

**Exercise 1.2** — Complete the function below to compute the **GC content** of a DNA sequence (the fraction of bases that are G or C). GC content is important because G-C base pairs are more stable than A-T pairs.

In [ ]:
def gc_content(sequence):
    """Return the GC content of a DNA sequence as a value between 0 and 1."""
    gc = sequence.count(___) + sequence.count(___)
    return gc / ___

print(gc_content(dna))  # Expected: ~0.567

### 1.3 Splitting a sequence into codons

**Exercise 1.3** — Use a loop to split `dna` into a **list of codons** (groups of 3 nucleotides).

In [ ]:
codons = []
for i in range(___, ___, ___):
    codon = dna[___:___]
    codons.append(codon)

print(codons)
# Expected: ['ATG', 'GTG', 'CAT', 'CTG', 'ACT', 'CCT', 'GAG', 'GAG', 'AAG', 'TCT']

### 1.4 Finding Open Reading Frames (ORFs)

A real genomic DNA sequence is not just coding sequence — it contains untranslated regions (UTRs), introns, and other non-coding stretches. To find the protein-coding region, we look for an **Open Reading Frame (ORF)**:

- Starts with **ATG** (the start codon, encoding Methionine)
- Ends with a **stop codon** (TAA, TAG, or TGA) in the same reading frame

A sequence can contain multiple ATGs — we need to find the correct one.

In [ ]:
# Example sequence with two possible ATG start sites
mystery = "TTAATGCGATAAATGGTGCATCTGACTCCTGAGGAGAAGTCTGAATAG"
#                ^-- ATG #1           ^-- ATG #2

**Exercise 1.4a** — Find all positions where `"ATG"` appears in `mystery`. Use the `.find()` method or a loop.

In [ ]:
stop_codons = ["TAA", "TAG", "TGA"]

atg_positions = []
for i in range(len(mystery)):
    if mystery[___:___] == ___:
        atg_positions.append(i)

print("ATG positions:", atg_positions)  # Expected: [3, 12]

**Exercise 1.4b** — For each ATG position, scan forward in steps of 3 to find the first in-frame stop codon. Print the ORF sequence and its length. Which is the correct ORF?

In [ ]:
for start in atg_positions:
    for i in range(start, len(mystery), 3):
        codon = mystery[i:i+3]
        if codon in ___:
            orf = mystery[start:i+3]
            print(f"ATG at {start}: ORF = {orf}  (length {len(orf)} bp)")
            break

# Expected:
# ATG at 3:  ORF = ATGCGATAA  (length 9 bp)   → encodes 2 amino acids
# ATG at 12: ORF = ATGGTGCATCTGACTCCTGAGGAGAAGTCTGAATAG  (length 36 bp)  → encodes 11 amino acids

---
## Section 2: From DNA to Protein

Cells read DNA in groups of 3 bases called **codons**. Each codon specifies one amino acid (or a stop signal). This is the **genetic code**.

The process:
1. **Transcription**: DNA → mRNA (T becomes U)
2. **Translation**: mRNA codons → amino acids → protein

### 2.1 Transcription: DNA → mRNA

In [ ]:
# The .replace() method substitutes one substring for another
mrna = dna.replace("T", "U")
print("mRNA:", mrna)

### 2.2 The codon table

We'll represent the genetic code as a Python dictionary mapping codons to amino acids (single-letter codes). Stop codons are represented as `"*"`.

In [ ]:
codon_table = {
    # Phenylalanine (F)
    "UUU": "F", "UUC": "F",
    # Leucine (L)
    "UUA": "L", "UUG": "L", "CUU": "L", "CUC": "L", "CUA": "L", "CUG": "L",
    # Isoleucine (I)
    "AUU": "I", "AUC": "I", "AUA": "I",
    # Methionine / Start (M)
    "AUG": "M",
    # Valine (V)
    "GUU": "V", "GUC": "V", "GUA": "V", "GUG": "V",
    # Serine (S)
    "UCU": "S", "UCC": "S", "UCA": "S", "UCG": "S", "AGU": "S", "AGC": "S",
    # Proline (P)
    "CCU": "P", "CCC": "P", "CCA": "P", "CCG": "P",
    # Threonine (T)
    "ACU": "T", "ACC": "T", "ACA": "T", "ACG": "T",
    # Alanine (A)
    "GCU": "A", "GCC": "A", "GCA": "A", "GCG": "A",
    # Tyrosine (Y)
    "UAU": "Y", "UAC": "Y",
    # Stop
    "UAA": "*", "UAG": "*", "UGA": "*",
    # Histidine (H)
    "CAU": "H", "CAC": "H",
    # Glutamine (Q)
    "CAA": "Q", "CAG": "Q",
    # Asparagine (N)
    "AAU": "N", "AAC": "N",
    # Lysine (K)
    "AAA": "K", "AAG": "K",
    # Aspartate (D)
    "GAU": "D", "GAC": "D",
    # Glutamate (E)
    "GAA": "E", "GAG": "E",
    # Cysteine (C)
    "UGU": "C", "UGC": "C",
    # Tryptophan (W)
    "UGG": "W",
    # Arginine (R)
    "CGU": "R", "CGC": "R", "CGA": "R", "CGG": "R", "AGA": "R", "AGG": "R",
    # Glycine (G)
    "GGU": "G", "GGC": "G", "GGA": "G", "GGG": "G",
}

print("Codons in table:", len(codon_table))

### 2.3 Translation

**Exercise 2.1** — Complete the `translate` function below. It should:
1. Convert the DNA sequence to mRNA (replace T with U)
2. Read the mRNA in codons of 3
3. Look up each codon in `codon_table`
4. Stop when it hits a stop codon (`"*"`)
5. Return the amino acid sequence as a string

In [ ]:
def translate(dna_sequence):
    """Translate a DNA sequence into a protein sequence."""
    mrna = dna_sequence.replace(___, ___)
    protein = ""
    for i in range(___, ___, ___):
        codon = mrna[___:___]
        amino_acid = codon_table.get(codon, "?")
        if amino_acid == ___:
            break
        protein += ___
    return protein

print(translate(dna))  # Expected: MVHLTPEEKS

---
## Section 3: Introducing Biopython

Biopython is a library that provides ready-made tools for biological sequence analysis. Let's see how it compares to what we just built — and use it to work with the real HBB coding sequence.

In [ ]:
# Install biopython if needed (run once)
# !pip install biopython

In [ ]:
from Bio.Seq import Seq

### 3.1 Bio.Seq — the Biopython sequence object

In [ ]:
seq = Seq(dna)

print("Complement:        ", seq.complement())
print("Reverse complement:", seq.reverse_complement())
print("Transcription:     ", seq.transcribe())
print("Translation:       ", seq.translate())

Notice that `seq.translate()` produces the same amino acid sequence as our manual `translate()` function.

**Exercise 3.1** — Use `seq.translate()` to translate `dna`, remove the trailing stop codon (`*`), and confirm it matches our manual result.

In [ ]:
bio_protein = str(seq.translate())
# Remove the trailing stop codon (*)
bio_protein_clean = bio_protein[___]

manual_result = translate(dna)
print("Biopython:", bio_protein_clean)
print("Manual:   ", manual_result)
print("Match:", bio_protein_clean == manual_result)

### 3.2 The full HBB coding sequence

Below is the complete coding sequence (CDS) of the human HBB gene, sourced from NCBI RefSeq accession **NM_000518**. It is 444 bp long and encodes a protein of 147 amino acids.

In [ ]:
# HBB CDS — NCBI RefSeq NM_000518
hbb_cds_str = (
    "ATGGTGCATCTGACTCCTGAGGAGAAGTCTGCCGTTACTGCCCTGTGGGGCAAGGTGAAC"
    "GTGGATGAAGTTGGTGGTGAGGCCCTGGGCAGGCTGCTGGTGGTCTACCCTTGGACCCAG"
    "AGGTTCTTTGAGTCCTTTGGGGATCTGTCCACTCCTGATGCTGTTATGGGCAACCCTAAG"
    "GTGAAGGCTCATGGCAAGAAAGTGCTCGGTGCCTTTAGTGATGGCCTGGCTCACCTGGAC"
    "AACCTCAAGGGCACCTTTGCCACACTGAGTGAGCTGCACTGTGACAAGCTGCACGTGGAT"
    "CCTGAGAACTTCAGGCTCCTGGGCAACGTGCTGGTCTGTGTGCTGGCCCATCACTTTGGC"
    "AAAGAATTCACCCCACCAGTGCAGGCTGCCTATCAGAAAGTGGTGGCTGGTGTGGCTAAT"
    "GCCCTGGCCCACAAGTATCACTAA"
)

print("CDS length:", len(hbb_cds_str), "bp")
print("First 30 bp:", hbb_cds_str[:30])

In [ ]:
# Translate the full CDS using Biopython
hbb_cds = Seq(hbb_cds_str)
hbb_protein = hbb_cds.translate(to_stop=True)
print("HBB protein (β-globin):")
print(hbb_protein)
print("Length:", len(hbb_protein), "amino acids")

**Exercise 3.2** — Use `.count()` on `hbb_protein` to find how many **glutamate (E)** residues are in β-globin. Then print the amino acid at position 7 of the CDS protein (0-based index 6) — this is the sickle cell mutation site.

In [ ]:
num_E = str(hbb_protein).count(___)
print("Glutamate (E) count:", num_E)
print("Amino acid at CDS position 7:", str(hbb_protein)[___])  # Expected: E

---
## Section 4: Mutations

A **mutation** is a change in a DNA sequence. Three important types:

| Type | Effect on protein | Example |
|---|---|---|
| **Synonymous** | No change (same amino acid) | GAG → GAA (both = E) |
| **Missense** | Different amino acid | GAG → GTG (E → V) |
| **Nonsense** | Premature stop codon | GAG → TAG (E → stop) |

The sickle cell mutation is a missense mutation: **GAG → GTG** (glutamate → valine).

> **Note on protein numbering:** In the literature, the sickle cell mutation is called "p.Glu**6**Val" — position **6** of β-globin. This counts from the first amino acid of the *mature* protein, after the initiator methionine (position 1 of the CDS) is cleaved. So "position 6 of mature protein" = codon **7** of the CDS.

### 4.1 Introducing a point mutation

Strings in Python are **immutable** — we can't change a character in place. Instead, we use slicing to build a new string.

In [ ]:
def point_mutation(sequence, position, new_base):
    """
    Introduce a single-nucleotide substitution.
    position: 0-based index in the sequence
    new_base: the replacement nucleotide (A, T, G, or C)
    """
    return sequence[:position] + new_base + sequence[position + 1:]

# Test on a short sequence
test = "ATGAAGCTT"
print("Original: ", test)
print("Mutated:  ", point_mutation(test, 4, "T"))  # A→T at position 4

### 4.2 The sickle cell mutation

CDS codon 7 (the sickle cell site) occupies positions **18–20** (0-based). The codon is **GAG** (Glu). The mutation changes position 19 (A → T): **GAG → GTG** (Glu → Val).

In [ ]:
# CDS codon 7 is at positions 18-20 (0-based)
print("CDS codon 7 (normal): ", hbb_cds_str[18:21])  # Should be GAG

# Sickle cell mutation: position 19, A → T
hbb_sickle = point_mutation(hbb_cds_str, 19, "T")
print("CDS codon 7 (sickle): ", hbb_sickle[18:21])   # Should be GTG

In [ ]:
# Translate both and compare
normal_protein = Seq(hbb_cds_str).translate(to_stop=True)
sickle_protein = Seq(hbb_sickle).translate(to_stop=True)

# Index 6 (0-based) = CDS codon 7 = mature protein position 6
print("Normal amino acid at position 7: ", normal_protein[6])   # E
print("Sickle amino acid at position 7: ", sickle_protein[6])   # V
print("Proteins otherwise identical?    ", normal_protein[7:] == sickle_protein[7:])

**Exercise 4.1** — Complete the function below to classify a mutation as synonymous, missense, or nonsense. Then test it on the sickle cell mutation.

In [ ]:
def classify_mutation(original_cds, mutated_cds):
    """
    Classify a point mutation by comparing the translated proteins.
    Returns 'synonymous', 'missense', or 'nonsense'.
    """
    original_protein = str(Seq(original_cds).translate())
    mutated_protein  = str(Seq(mutated_cds).translate())

    # Check for nonsense: a stop codon (*) appears earlier in the mutated protein
    if ___ in mutated_protein and mutated_protein.index(___) < original_protein.index(___):
        return "nonsense"
    # Check for missense: proteins differ
    elif original_protein != ___:
        return "missense"
    # Otherwise: synonymous
    else:
        return ___

result = classify_mutation(hbb_cds_str, hbb_sickle)
print("Sickle cell mutation is:", result)  # Expected: missense

**Exercise 4.2** — Use `point_mutation` and `classify_mutation` to:
1. Create a **synonymous** mutation somewhere in the HBB CDS
2. Create a **nonsense** mutation (change a codon to a stop codon)

Print the position, the nucleotide change, and the classification.

In [ ]:
# Your code here


**Exercise 4.3 (Open-ended)** — The table below lists three real HBB variants. For each one:
- Introduce the mutation using `point_mutation`
- Classify it using `classify_mutation`
- Describe in one sentence what disease or effect it causes

| Variant | CDS position (0-based) | Change | Expected classification |
|---|---|---|---|
| HbC | 18 | G → A | missense (E → K) |
| HbE | 78 | G → A | missense (E → K) |
| HbD | 363 | G → C | missense (E → Q) |

In [ ]:
# Your code here


---
## Section 5: Advanced Topics (Self-Study)

These exercises are for students who want to go further. They are not required.

### 5.1 Fetching sequences from NCBI

We hardcoded the HBB CDS above, but Biopython can fetch sequences directly from NCBI's database. This is useful for any gene.

**Challenge:** Use `Bio.Entrez` to fetch the full HBB mRNA record (accession `NM_000518`) and extract the CDS automatically from the GenBank features. Confirm it matches `hbb_cds_str`.

In [ ]:
# Your code here
# Hint:
# from Bio import Entrez, SeqIO
# Entrez.email = "your.email@example.com"
# handle = Entrez.efetch(db="nucleotide", id="NM_000518", rettype="gb", retmode="text")
# record = SeqIO.read(handle, "genbank")

### 5.2 HGVS variant notation

In clinical genetics, variants are reported in **HGVS notation**, e.g., `c.20A>T` means CDS position 20 (1-based), A → T. This is the standard way to report the sickle cell mutation.

**Challenge:** Write a function `parse_hgvs(hgvs_string)` that parses a string like `"c.20A>T"` and returns the 0-based position, the reference base, and the alternate base. Apply it to introduce the sickle cell mutation programmatically.

In [ ]:
# Your code here


### 5.3 Motif finding

Transcription factors bind to specific short DNA sequences called **motifs**. For example, the TATA box (`TATAAAA`) is found in many gene promoters.

**Challenge:** Write a function `find_motif(sequence, motif)` that returns all start positions (0-based) where `motif` appears, where `N` in the motif matches any base. Use it to search the 500 bp upstream of the HBB CDS for the motif `"CCWWGG"` (W = A or T), a known binding site.

*Hint: the `re` module supports character classes like `[AT]`.*

In [ ]:
# Your code here


---
## Where Do We Go From Here?

Today we worked entirely within the **coding sequence** — the part of the genome that gets translated into protein. But genes are regulated by much more than their CDS.

### Gene Regulation

The same genome is present in every cell of your body, yet a neuron looks and behaves very differently from a red blood cell. The difference is **gene regulation** — which genes are turned on or off, and by how much.

Key regulatory elements:
- **Enhancers** — distant DNA sequences (sometimes thousands of bases away) that boost transcription of a gene
- **Promoters** — sequences just upstream of a gene where the transcription machinery binds
- **Epigenetic modifications** — chemical marks on DNA or histones (the proteins DNA wraps around) that control whether a region is accessible for transcription
- **Chromatin structure** — how tightly DNA is packaged differs between cell types and disease states, controlling which genes can be read

Even a mutation *outside* the coding sequence — in an enhancer or promoter — can cause disease by changing how much of a protein is made.

### Measuring Gene Activity: RNA-seq

How do we know which genes are active in a cell? One powerful approach is **RNA sequencing (RNA-seq)**: extract all the RNA from a sample, sequence it, and count how many copies of each transcript are present. This gives a genome-wide snapshot of gene expression.

RNA-seq data looks very different from a simple DNA sequence — instead of one string, you have millions of short reads that need to be mapped, counted, and compared statistically. That's what we'll tackle in **Lecture 2**.